In [2]:
# =============================
# IMPORT LIBRARIES
# =============================
import nltk
import string
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer

# =============================
# DOWNLOAD NLTK DATA
# =============================
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')  # improves lemmatization
nltk.download('punkt_tab') # Added to resolve the LookupError

# =============================
# INPUT TEXT (NO FILE NEEDED)
# =============================
text = """
Music is an art form that uses sound organized in time.
It includes elements such as rhythm, melody, and harmony.
Music is found in every culture around the world.
People use music for entertainment, expression, and communication.
Different genres of music include classical, jazz, pop, and hip-hop.
Technology has changed how music is produced and consumed.
Streaming platforms allow people to access music easily.
Music can influence emotions and bring people together.
"""

# =============================
# TOKENIZATION
# =============================
sentences = sent_tokenize(text)

# =============================
# PREPROCESSING
# =============================
stop_words = set(stopwords.words('english'))
punctuation = set(string.punctuation)
lemmatizer = WordNetLemmatizer()

def preprocess_word(word):
    if word in stop_words or word in punctuation:
        return None
    return lemmatizer.lemmatize(word, "v")

def preprocess_sentence(sentence):
    words = word_tokenize(sentence.lower())
    processed = []
    for word in words:
        w = preprocess_word(word)
        if w is not None:
            processed.append(w)
    return ' '.join(processed)

corpus = [preprocess_sentence(sentence) for sentence in sentences]

# =============================
# TF-IDF + COSINE SIMILARITY
# =============================
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)
similarity_matrix = (tfidf_matrix * tfidf_matrix.T).toarray()

# =============================
# BUILD GRAPH
# =============================
G = nx.Graph()

for i in range(len(sentences)):
    G.add_node(i, sentence=sentences[i])

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        weight = similarity_matrix[i][j]
        if weight > 0:  # optional: ignore zero similarity
            G.add_edge(i, j, weight=weight)

# =============================
# DRAW GRAPH (OPTIONAL)
# =============================
plt.figure(figsize=(8, 6))
nx.draw(G, with_labels=True)
plt.savefig("graph.png")
plt.close()

# =============================
# APPLY PAGERANK
# =============================
pagerank_scores = nx.pagerank(G, weight='weight')

# Sort sentences by importance
ranked_sentences = sorted(pagerank_scores.items(), key=lambda x: x[1], reverse=True)

# =============================
# GENERATE SUMMARY
# =============================
top_n = 5
summary = [sentences[i] for i, _ in ranked_sentences[:top_n]]

# =============================
# OUTPUT RESULTS
# =============================
print("\n================ ORIGINAL TEXT ================\n")
print(text)

print("\n================ SUMMARY ================\n")
for sentence in summary:
    print("-", sentence)

# ======================
# SAVE SUMMARY TO FILE
# ======================
with open("output.txt", "w") as f:
    for sentence in summary:
        f.write(sentence + "\n")

print("\n✅ Summary saved to output.txt")
print("✅ Graph saved as graph.png")


================ ORIGINAL TEXT ================


Music is an art form that uses sound organized in time.
It includes elements such as rhythm, melody, and harmony.
Music is found in every culture around the world.
People use music for entertainment, expression, and communication.
Different genres of music include classical, jazz, pop, and hip-hop.
Technology has changed how music is produced and consumed.
Streaming platforms allow people to access music easily.
Music can influence emotions and bring people together.


================ SUMMARY ================

- People use music for entertainment, expression, and communication.
- Music can influence emotions and bring people together.
- Streaming platforms allow people to access music easily.
- Different genres of music include classical, jazz, pop, and hip-hop.
- 
Music is an art form that uses sound organized in time.

✅ Summary saved to output.txt
✅ Graph saved as graph.png


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
